In [ ]:
import tables_io
import hpmcm
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
redo_rubin_md = False

our_geom = hpmcm.shear_utils.ShearCellGeometry(ref_dir=(340.5000000000, -9.6694214876))

rubin_shear_match_files = [
    '8006/shear_object_reduced_uncleaned_8006_ns.parq',
    '8006/shear_object_reduced_uncleaned_8006_1m.parq',
    '8006/shear_object_reduced_uncleaned_8006_1p.parq',
    '8006/shear_object_reduced_uncleaned_8006_2m.parq',
    '8006/shear_object_reduced_uncleaned_8006_2p.parq',    
]
rubin_shear_match_ids = list(range(len(rubin_shear_match_files)))

if redo_rubin_md:
    hpmcm.shear_utils.splitRubinMDByTypeAndClean(
        '8006/shear_object_8006.parq',
        8006,
        0.01,
    )

    for st in ['ns', '1p', '1m', '2p', '2m']:
        hpmcm.utils.reduceRubinMDTable(
            f"8006/shear_object_8006_uncleaned_8006_{st}.parq",
            f"8006/shear_object_reduced_uncleaned_8006_{st}.parq",
        )

In [ ]:
rubin_shear_match = hpmcm.shear_match.ShearMatch.createShearMatch(
    geometry=our_geom,
)

rubin_shear_match.reduceData(rubin_shear_match_files, rubin_shear_match_ids)
rubin_shear_match.analysisLoop()
rubin_stats = rubin_shear_match.extractStats()
rubin_stats.update(**rubin_shear_match.extractShearStats())
tables_io.write(rubin_stats, '8006/rubin_md_match_')


In [ ]:
cluster_assoc = hpmcm.output_tables.ClusterAssocTable.read('8006/rubin_md_match_cluster_assoc.pq', [])
object_assoc = hpmcm.output_tables.ObjectAssocTable.read('8006/rubin_md_match_object_assoc.pq', [])
cluster_stats = hpmcm.output_tables.ClusterStatsTable.read('8006/rubin_md_match_cluster_stats.pq', [])
object_stats = hpmcm.output_tables.ObjectStatsTable.read('8006/rubin_md_match_object_stats.pq', [])

rubin_shear_match_r = hpmcm.shear_match.ShearMatch.createShearMatch(
    geometry=our_geom,
)
rubin_shear_match_r.populateFromTables(
    cluster_assoc, object_assoc, cluster_stats, object_stats, rubin_shear_match_files, rubin_shear_match_ids,
)

In [ ]:
rubin_shear_match_r.cell_dict[202]

In [ ]:
rubin_shear_match_r.save('8006/rubin_md_shear_match')

In [ ]:
rubin_shear_match_r2 = hpmcm.ShearMatch.load('8006/rubin_md_shear_match')

In [ ]:
redo_desc_md = False
desc_shear_match = hpmcm.shear_match.ShearMatch.createShearMatch(
    geometry=our_geom,
)
desc_shear_match_files = [
    '8006/mdet-run-reduced_uncleaned_8006_ns.parq',
    '8006/mdet-run-reduced_uncleaned_8006_1m.parq',
    '8006/mdet-run-reduced_uncleaned_8006_1p.parq',
]
desc_shear_match_ids = list(range(len(desc_shear_match_files)))

if redo_desc_md:
    hpmcm.shear_utils.splitDESCMDByTypeAndClean(
        '8006/mdet-run-8006.parq',
        8006,
        0.01,
    )

    for st in ['ns', '1p', '1m']:
        hpmcm.utils.reduceDESCMDTable(
            f"8006/mdet-run-8006_uncleaned_8006_{st}.parq",
            f"8006/mdet-run-reduced_uncleaned_8006_{st}.parq",
        )

desc_shear_match.reduceData(desc_shear_match_files, desc_shear_match_ids)
desc_shear_match.analysisLoop()
desc_stats = desc_shear_match.extractStats()
desc_stats.update(**desc_shear_match.extractShearStats())
tables_io.write(desc_stats, '8006/desc_md_match_')    

In [ ]:
all_files = [
    '8006/object_reduced_8006.parq',
    '8006/anacal_reduced_8006.parq',
    '8006/desc_md_match_object_stats.pq',
    '8006/rubin_md_match_object_stats.pq',    
]
all_ids = list(range(len(all_files)))
pixel_scale = 0.2
n_pix_region = 30000
region_size_deg = n_pix_region*pixel_scale/3600
wcs_match = hpmcm.WcsMatch.create(
    ref_dir=(340.5000000000, -9.6694214876),
    region_size=(region_size_deg, region_size_deg),
    pixel_size=pixel_scale/3600,
    cell_size=150,
    cell_buffer=5,
    cell_max_object=10000,
)
wcs_match.reduceData(all_files, all_ids)
wcs_match.analysisLoop()
obj_stats = wcs_match.extractStats()
tables_io.write(obj_stats, '8006/obj_match_')


In [ ]:
cell_data_rubin = rubin_shear_match.cell_dict[20200]
cluster_ids_rubin = list(cell_data_rubin.cluster_dict.keys())
cell_data_desc = desc_shear_match.cell_dict[20200]
cluster_ids_desc = list(cell_data_desc.cluster_dict.keys())
cell_data_obj = wcs_match.cell_dict[20200]
cluster_ids_obj = list(cell_data_obj.cluster_dict.keys())

color_dict = {
    0:'black',
    1:'blue',
    2:'green',
    3:'cyan',
    4:'red',
}

In [ ]:
(obj_stats['cluster_stats'].catalog_mask == 15).sum()

In [ ]:
obj_stats['cluster_stats']

In [ ]:
clusters = obj_stats['cluster_stats']
bins = np.logspace(0, 5, 51)
for i_bit, label in enumerate(['obj', 'anacal', 'rubin_md', 'desc_md']):
    if i_bit in [3]:
        continue
    mask_val = 1 << i_bit
    mask = np.bitwise_and(clusters.catalog_mask, mask_val) == mask_val
    _ = plt.hist(clusters.snr[mask], bins=bins, alpha=0.2, label=label)
_ = plt.xscale('log')
_ = plt.yscale('log')
_ = plt.legend()

In [ ]:
desc_stats['cluster_stats']

In [ ]:
_ = plt.scatter(obj_stats['cluster_stats'].x_pix, obj_stats['cluster_stats'].ra, s=1)
_ = plt.scatter(rubin_stats['cluster_stats'].x_pix, rubin_stats['cluster_stats'].ra, s=1)
_ = plt.scatter(desc_stats['cluster_stats'].x_pix, desc_stats['cluster_stats'].ra, s=1)

In [ ]:
_ = plt.scatter(obj_stats['cluster_stats'].y_pix, obj_stats['cluster_stats'].dec, s=1)
_ = plt.scatter(rubin_stats['cluster_stats'].y_pix, rubin_stats['cluster_stats'].dec, s=1)
_ = plt.scatter(desc_stats['cluster_stats'].y_pix, desc_stats['cluster_stats'].dec, s=1)

In [ ]:
cluster_rubin = cell_data_rubin.cluster_dict[cluster_ids_rubin[0]]
cluster_desc = cell_data_desc.cluster_dict[cluster_ids_desc[4]]

In [ ]:
_ = plt.scatter(obj_stats['cluster_stats'].ra, obj_stats['cluster_stats'].dec, s=4)
_ = plt.scatter(rubin_stats['cluster_stats'].ra, rubin_stats['cluster_stats'].dec, s=2)
_ = plt.scatter(desc_stats['cluster_stats'].ra, desc_stats['cluster_stats'].dec, s=1)

In [ ]:
_ = plt.scatter(cluster_rubin.x_pix, cluster_rubin.y_pix, s=1)
_ = plt.scatter(rubin_stats['cluster_stats'].x_pix, rubin_stats['cluster_stats'].y_pix)

In [ ]:
cluster_rubin.data

In [ ]:
cluster_desc.data

In [ ]:
_ = hpmcm.viz_utils.showCluster(
    cell_data_rubin.countsMap(), 
    cluster_rubin,
    cell_data_rubin,
    None,
    color_dict,
)

In [ ]:
_ = hpmcm.viz_utils.showCluster(
    cell_data_desc.countsMap(), 
    cluster_desc,
    cell_data_desc,
    None,
    color_dict,
)

In [ ]:
cluster.x_cent

In [ ]:
mask = rubin_stats['cluster_stats'].cell_idx == 20100
cell_clusters = rubin_stats['cluster_stats'][mask]
cell_clusters

In [ ]:
cell_clusters.x_pix - cell_clusters.x_cent